In [12]:
import pandas as pd
import os
import numpy as np
import sys
sys.path.append("../")
from bin.constants import ISOPEP_DOMAINS
from dotenv import load_dotenv
load_dotenv("../.env")

DISULFIDE_BONDS_TABLE = os.getenv("DISULFIDE_BONDS_TABLE")
AFDB_JESS_SCAN_TABLE = os.getenv("AFDB_JESS_SCAN_TABLE")

dis_df = pd.read_csv(DISULFIDE_BONDS_TABLE)
str_df = pd.read_csv(AFDB_JESS_SCAN_TABLE, low_memory=False)


In [13]:
table_df = dis_df.drop_duplicates(["uniprot_acc", "seq_start"]) \
    .value_counts(["pfamA_id", "pfamA_acc", "clan_id", "phylum", "kingdom"]).reset_index()
table_df["Pfam ID (accession code)"] = table_df.apply(lambda x: f"{x['pfamA_id']} ({x['pfamA_acc']})", axis=1)
count_df = str_df[str_df["pfamA_id"].isin(ISOPEP_DOMAINS)].drop_duplicates(["uniprot_acc", "seq_start"]) \
    .value_counts("pfamA_id").reset_index().rename(columns={"count":'total_count'})

table_df = pd.merge(table_df, count_df, how='left', on="pfamA_id")
table_df.loc[table_df["kingdom"]=="Archaea", "phylum"] = "Archaea"

# Calculate 'other' counts
phyla_to_keep = table_df.query('count>100')["phylum"].unique().tolist()
pfam_to_keep = table_df.query('count>100')["Pfam ID (accession code)"].unique().tolist()
#phyla_to_keep.pop("ND")
other = (
    table_df[~table_df["phylum"].isin(phyla_to_keep)]
    .groupby("Pfam ID (accession code)")
    .agg({"count": "sum"})
    .reset_index()
)
other["phylum"] = "other"
# Add total_count to 'other'
other = pd.merge(other, table_df[["Pfam ID (accession code)", "total_count"]].drop_duplicates(), on="Pfam ID (accession code)", how="left")

# Calculate 'total' counts
total = (
    table_df.groupby("Pfam ID (accession code)")
    .agg({"count": "sum", "total_count": "first"})
    .reset_index()
)
total["phylum"] = "total"

table_df = pd.concat([table_df, other, total], ignore_index=True)
# Keep only phyla_to_keep, other, and total
keep_phyla = phyla_to_keep + ["other", "total"]

# Aggregate to ensure uniqueness
agg_df = (
    table_df[(table_df["phylum"].isin(keep_phyla))&(table_df["Pfam ID (accession code)"].isin(pfam_to_keep))]
    .groupby(["Pfam ID (accession code)", "phylum"], as_index=False)
    .agg({"count": "sum", "total_count": "first"})
)

pivot = (
    agg_df.sort_values(["Pfam ID (accession code)", "count"])
    .assign(
        perc=lambda df: (df["count"] / df["total_count"] * 100).round(1),
        count_perc=lambda df: df["count"].astype(str) + " (" + df["perc"].astype(str) + "%)"
    )
    .pivot(index="Pfam ID (accession code)", columns="phylum", values="count_perc")
    .fillna("")
)

pivot

phylum,Acidobacteriota,Actinomycetota,Chloroflexota,Pseudomonadota,Archaea,other,total
Pfam ID (accession code),,,,,,,
DUF11 (PF01345),164 (1.3%),1080 (8.8%),382 (3.1%),838 (6.8%),221 (1.8%),756 (6.1%),3441 (27.9%)
DUF5979 (PF19407),,747 (33.7%),4 (0.2%),30 (1.4%),,2 (0.1%),783 (35.3%)
DUF7507 (PF24346),16 (0.5%),628 (18.3%),67 (1.9%),158 (4.6%),84 (2.4%),127 (3.7%),1080 (31.4%)
DUF7927 (PF25549),,171 (69.2%),,7 (2.8%),,1 (0.4%),179 (72.5%)
FctA (PF12892),,106 (4.6%),,,,3 (0.1%),109 (4.7%)
GramPos_pilinD1 (PF16555),,199 (9.8%),,,,,199 (9.8%)
SpaA (PF17802),1 (0.0%),1425 (7.5%),21 (0.1%),9 (0.0%),5 (0.0%),46 (0.2%),1507 (8.0%)
SpaA_4 (PF24514),22 (4.9%),120 (26.5%),27 (6.0%),138 (30.5%),7 (1.5%),54 (11.9%),368 (81.4%)
